# Капстоун · ноутбук 3. Инференс

*Курс «Modern LLMs» · итоговый практикум*

Теперь исследуем работу обученной модели: отдельно измерим prefill и decode, посчитаем память KV-кэша и проверим повторное использование префикса. Затем сравним приближённые форматы весов и выполним спекулятивное декодирование. М10 и М11 дают формулы и условия, с которыми мы сопоставим опыт.

Читаем три вида результатов отдельно: геометрические равенства, изменения распределения ответов и времена данного запуска. Для квантизации используем синтетический корпус с широким словарём и несколько метрик. Его сложность не гарантирует заметной деградации, а малый loss не гарантирует её отсутствия.

Сеть не нужна. Ноутбук использует CPU или доступный GPU; длительность зависит от устройства и бюджета. Первый проход выполняет все ячейки по порядку. Во втором измените одну настройку и объясните, какое измерение должно на неё отреагировать.

## 1. Окружение

In [ ]:
import importlib
import sys
import subprocess
import zipfile
from pathlib import Path

ARCHIVE_GLOB = "modern-llms-capstone-*.zip"

def _activate_capstone_root(candidate: Path):
    package_init = candidate / "minillm" / "__init__.py"
    if not package_init.exists():
        return None
    root = str(candidate.resolve())
    if root not in sys.path:
        sys.path.insert(0, root)
    return root

def _find_capstone_root(search_root: Path):
    direct = _activate_capstone_root(search_root)
    if direct:
        return direct
    for child in sorted(search_root.glob("modern-llms-capstone-*")):
        found = _activate_capstone_root(child)
        if found:
            return found
    return None

def _safe_extract(archive_path: Path, destination: Path):
    destination = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            member_path = Path(member.filename)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise RuntimeError(f"unsafe path in archive: {member.filename}")
        archive.extractall(destination)

def ensure_minillm():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        found = _find_capstone_root(candidate)
        if found:
            return "пакет проекта: " + found

    try:
        import minillm  # noqa: F401
        return "minillm уже доступен"
    except ImportError:
        pass

    archives = sorted(Path.cwd().glob(ARCHIVE_GLOB))
    if not archives:
        try:
            from google.colab import files
        except ImportError:
            files = None
        if files is not None:
            print("Загрузите ZIP-пакет Capstone из урока Stepik. Структура каталогов будет восстановлена автоматически.")
            files.upload()
            archives = sorted(Path.cwd().glob(ARCHIVE_GLOB))

    if archives:
        archive_path = archives[-1]
        destination = Path.cwd() / ".capstone_runtime"
        destination.mkdir(exist_ok=True)
        _safe_extract(archive_path, destination)
        matches = sorted(destination.glob("*/minillm/__init__.py"))
        if len(matches) != 1:
            raise RuntimeError(
                f"expected one minillm package in {archive_path.name}, "
                f"found {len(matches)}"
            )
        root = _activate_capstone_root(matches[0].parent.parent)
        if root:
            return "архив проекта: " + str(archive_path)

    raise RuntimeError("minillm не найден. Скачайте языковой ZIP-пакет Capstone из урока Stepik и либо распакуйте его целиком, либо загрузите ZIP в текущую сессию Colab.")

print(ensure_minillm())

for package in ("torch", "numpy", "matplotlib", "regex"):
    try:
        importlib.import_module(package)
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", package],
            check=True,
        )

import copy, math, os, random, time
import torch, numpy as np
import matplotlib.pyplot as plt

SEED = 0
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
from minillm.precision import autocast_context, model_parameter_dtype, resolve_compute_dtype
COMPUTE_DTYPE = resolve_compute_dtype(DEVICE, "auto")
print("torch", torch.__version__, "| устройство:", DEVICE, "| вычисления:", COMPUTE_DTYPE)

### 1.1. Передача модели из ноутбука 2

При последовательном запуске в одном каталоге откроем `runs/posttrain/final` и выполним генерацию сохранённой моделью. Если артефакта нет, ячейка сообщит об этом. Дальнейшие сравнения используют отдельную модель и общий набор входов: так постобучение не становится скрытой переменной опыта с квантизацией.

In [ ]:
from minillm.chat import ChatTemplate, Message
from minillm.generate import SamplingConfig, generate
from minillm.tokenizer import BBPETokenizer
from minillm.train import load_model_artifact

POSTTRAIN_ARTIFACT = Path("runs/posttrain/final")
if (POSTTRAIN_ARTIFACT / "model.pt").exists():
    handoff_model, handoff_meta = load_model_artifact(POSTTRAIN_ARTIFACT, DEVICE)
    handoff_tokenizer = BBPETokenizer.load(POSTTRAIN_ARTIFACT / "tokenizer.json")
    handoff_template = ChatTemplate()
    handoff_ids, _ = handoff_template.encode_with_mask(
        [Message("system", "Ты отвечаешь кратко и по делу."),
         Message("user", "Что делает токенизатор?")],
        handoff_tokenizer, add_generation_prompt=True)
    handoff_out, handoff_stats = generate(
        handoff_model, handoff_ids, 40, SamplingConfig(temperature=0.0),
        stop_tokens={handoff_tokenizer.specials["<|eot_id|>"]})
    print("post-training handoff:", handoff_meta)
    print("ответ:", repr(handoff_tokenizer.decode(handoff_out[len(handoff_ids):])))
    print("KV bytes:", handoff_stats.kv_bytes)
else:
    print("post-training артефакт не найден — self-contained маршрут продолжается")

## 2. Корпус для сравнений

Соберём псевдоязык из случайных слов с неравномерными частотами. Общий закон генерации известен, но конкретные последовательности различаются. Это позволяет сравнить модели на контролируемых входах; оно не исключает запоминания конечного обучающего набора.

Данные для проверки отделим до обучения токенизатора и модели. Метрики на них показывают изменение предсказаний в данной синтетической задаче. Для выводов о переводе, коде или полезности ответов понадобятся соответствующие задачи.

In [ ]:
from minillm.tokenizer import BBPETokenizer
from minillm.data import TokenStream

def high_entropy_corpus(n_lines: int = 12_000, lexicon: int = 600,
                        seed: int = 0) -> str:
    """Псевдоязык: большой лексикон, зипфовское распределение частот.

    Неравномерные частоты задают контролируемую задачу предсказания.
    Loss измеряется на BPE-токен, поэтому энтропия распределения слов
    не является готовым эталоном для этой величины.
    """
    rng = random.Random(seed)
    alphabet = "абвгдеиклмнопрстуфхцчшы"
    words = ["".join(rng.choice(alphabet) for _ in range(rng.randint(3, 8)))
             for _ in range(lexicon)]
    weights = [1.0 / (i + 1) ** 0.9 for i in range(lexicon)]   # зипфовский хвост
    lines = []
    for _ in range(n_lines):
        k = rng.randint(6, 14)
        lines.append(" ".join(rng.choices(words, weights=weights, k=k)) + ".\n")
    return "".join(lines)

corpus = high_entropy_corpus(n_lines=8_000, seed=SEED)
lines = corpus.splitlines(keepends=True)
cut = int(len(lines) * 0.9)
train_text, eval_text = "".join(lines[:cut]), "".join(lines[cut:])
tokenizer = BBPETokenizer.train(train_text, vocab_size=1024, specials=["<|eos|>"])
stream = TokenStream.from_text(train_text, tokenizer, seq_len=128)
eval_stream = TokenStream.from_text(eval_text, tokenizer, seq_len=128)
print(f"корпус {len(corpus):,} символов · словарь {tokenizer.vocab_size} · "
      f"{len(stream):,} окон · sha {stream.checksum()}")
print(f"loss равномерной модели: ln(V) = "
      f"{math.log(tokenizer.vocab_size):.3f}")

## 3. Две модели: target и черновик

Спекулятивному декоду нужны две модели одного словаря: большая, чьё
распределение мы обязаны сохранить, и маленькая, которая дёшево предлагает
кандидатов. Обучаем обе на одних данных — так черновик оказывается похож на
target, а от похожести напрямую зависит доля приёма (M11).

In [ ]:
from minillm.config import MiniLLMConfig
from minillm.model import MiniLLM
from minillm.train import TrainConfig, evaluate, train

def build(hidden: int, layers: int, heads: int, kv: int) -> MiniLLM:
    cfg = MiniLLMConfig.profile("S", vocab_size=tokenizer.vocab_size,
                                hidden_size=hidden, num_hidden_layers=layers,
                                num_attention_heads=heads, num_key_value_heads=kv,
                                head_dim=hidden // heads, max_position_embeddings=256)
    return MiniLLM(cfg).to(DEVICE)

# Бюджет рассчитан на CPU. Увеличение числа шагов — отдельный опыт:
# после него заново измеряются loss и чувствительность к квантизации.
models = {}
for name, (hidden, layers, heads, kv, steps) in {
    "target": (192, 4, 6, 2, 250),
    "draft":  (96, 2, 4, 2, 250),
}.items():
    if os.getenv("CAPSTONE_FAST_VALIDATION") == "1":
        steps = 12
    torch.manual_seed(SEED)
    model = build(hidden, layers, heads, kv)
    print(f"--- {name}: {model.num_parameters():,} параметров")
    train(model, stream, TrainConfig(steps=steps, seq_len=128, micro_batch_size=8,
                                     optimizer="muon", seed=SEED, device=DEVICE, compute_dtype="auto",
                                     log_every=steps // 2))
    models[name] = model

target, draft = models["target"], models["draft"]
eval_cfg = TrainConfig(seq_len=128, micro_batch_size=8, device=DEVICE, compute_dtype="auto")
base_loss = evaluate(target, eval_stream, eval_cfg, batches=(2 if os.getenv("CAPSTONE_FAST_VALIDATION") == "1" else 16))
BASELINE_COMPUTE = resolve_compute_dtype(DEVICE, "auto")
BASELINE_PARAMETER = model_parameter_dtype(target)
BASELINE_LABEL = f"{BASELINE_PARAMETER}/{BASELINE_COMPUTE}"
BASELINE_STORAGE_BITS = 32.0 if BASELINE_PARAMETER == "fp32" else 16.0
print("target eval loss:", base_loss, "uniform reference:", math.log(tokenizer.vocab_size))

## 4. Prefill и decode: что измеряет время

Prefill обрабатывает позиции промпта совместно; при decode кэш позволяет не пересчитывать прежние K/V. В плотной модели с малым батчем чтение весов может ограничивать скорость decode, тогда как длинный prefill часто лучше использует вычислительные блоки. М10 выводит условия через арифметическую интенсивность.

В этой реализации малые матрицы, накладные расходы Python и работа кэша тоже влияют на время. Зависимость задержки от длины сама по себе не доказывает аппаратное узкое место. Зафиксируем устройство, типы и длины, затем сравним измеренные фазы; для диагноза потребовался бы профилировщик.

In [ ]:
from minillm.generate import SamplingConfig, generate

rows = []
for prompt_len in (16, 32, 64, 128):
    prompt = tokenizer.encode(corpus[:4000])[:prompt_len]
    _, stats = generate(target, prompt, 32, SamplingConfig(temperature=0.0))
    rows.append((prompt_len, stats.prefill_seconds * 1000,
                 stats.decode_seconds / stats.generated_tokens * 1000,
                 stats.kv_bytes / 1024))

print(f"{'промпт':>7} {'prefill, мс':>12} {'мс/токен decode':>17} {'KV-кэш, KiB':>13}")
for prompt_len, prefill_ms, decode_ms, kv_kib in rows:
    print(f"{prompt_len:>7} {prefill_ms:>12.2f} {decode_ms:>17.2f} {kv_kib:>13.1f}")

print('Сравните измеренные фазы; для определения узкого места нужны повторности и профилирование.')

## 5. KV-кэш: арифметика и три варианта внимания

Формула из M5: `2 · L · H_kv · d_head · sizeof(dtype)` байт на токен. Здесь
она не цитируется, а сверяется с фактически выделенной памятью.

In [ ]:
from minillm.config import gqa_variants, kv_bytes_per_token

variants = gqa_variants(target.cfg)
runtime_bytes = 4 if BASELINE_COMPUTE == "fp32" else 2
print(f"{'схема':>6} {'H_kv':>5} {('байт/токен ' + BASELINE_COMPUTE):>20} {'на 128K контекста':>20}")
for name, cfg in variants.items():
    per_token = kv_bytes_per_token(cfg, dtype_bytes=runtime_bytes)
    print(f"{name:>6} {cfg.num_key_value_heads:>5} {per_token:>20,} "
          f"{per_token * 131072 / 1024 ** 3:>18.2f} ГиБ")

# Сверка формулы с фактом на нашей модели.
prompt = tokenizer.encode(corpus[:2000])[:48]
cache = target.new_cache()
with torch.no_grad():
    with autocast_context(DEVICE, BASELINE_COMPUTE):
        target(torch.tensor([prompt], device=DEVICE), cache=cache)
actual_bytes = cache.nbytes() // (len(prompt) * 2 * target.cfg.num_hidden_layers * target.cfg.kv_dim)
expected = kv_bytes_per_token(target.cfg, dtype_bytes=actual_bytes) * len(prompt)
print(f"\nфактический dtype кэша: {str(cache.dtype()).replace('torch.', '')}")
print(f"факт {cache.nbytes():,} Б == формула {expected:,} Б: {cache.nbytes() == expected}")

assert cache.nbytes() == expected

### Префиксный кэш

Повторяющийся системный префикс можно вычислить один раз и сохранить его K/V. Для каждого запроса берём отдельную копию кэша: хвост одного запроса не должен попасть в другой. В опыте используется жадный выбор, а позиции, маска и веса остаются прежними.

Проверяем совпадение полученных токенов в текущем запуске. Эквивалентность логитов понимается с численным допуском: почти равные лидеры могут поменяться местами при округлении. При выборке потребовалось бы согласовать ещё и случайность; одинаковое распределение не гарантирует одинаковую отдельную последовательность.

In [ ]:
system_prompt = tokenizer.encode(corpus[:1200])[:96]
questions = [tokenizer.encode(corpus[5000 + 500 * i:5000 + 500 * i + 200])[:16]
             for i in range(4)]

warm = target.new_cache()
if DEVICE.startswith("cuda"):
    torch.cuda.synchronize()
t0 = time.perf_counter()
with torch.no_grad():
    with autocast_context(DEVICE, BASELINE_COMPUTE):
        target(torch.tensor([system_prompt], device=DEVICE), cache=warm)
if DEVICE.startswith("cuda"):
    torch.cuda.synchronize()
warmup_ms = (time.perf_counter() - t0) * 1000

cold_total = hot_total = 0.0
identical = True
for tail in questions:
    cold, cold_stats = generate(target, system_prompt + tail, 16,
                                SamplingConfig(temperature=0.0))
    hot, hot_stats = generate(target, system_prompt + tail, 16,
                              SamplingConfig(temperature=0.0), cache=warm.clone())
    identical &= cold == hot
    cold_total += cold_stats.prefill_seconds
    hot_total += hot_stats.prefill_seconds

print(f"результаты совпадают: {identical}")
print(f"prefill без кэша:  {cold_total * 1000:6.1f} мс на {len(questions)} запросов")
print(f"prefill с кэшем:   {hot_total * 1000:6.1f} мс (+{warmup_ms:.1f} мс прогрева один раз)")
print(f"экономия при {len(questions)} запросах: "
      f"×{cold_total / max(hot_total, 1e-9):.1f} на фазе prefill")

## 6. Квантизация и три вида расхождения

Сначала посчитаем хранение: для полной группы из $g$ четырёхбитных весов с восьмибитным масштабом получается $(4g+8)/g$ бит на вес — 4,25 при $g=32$ и 4,50 при $g=16$. Реальный отчёт учитывает также неполные группы и метаданные. Реализации MXFP4/NVFP4 здесь учебные приближения.

Loss оценивает вероятность правильных токенов, KL — отличие всего распределения от исходной модели, top-1 agreement — совпадение наиболее вероятного токена. Метрики отвечают на разные вопросы; их чувствительность зависит от входов и ошибок. Даже заметное изменение top-1 не переводится автоматически в заметность для пользователя, поэтому конечную задачу проверяют отдельно.

In [ ]:
from minillm.generate import quantize_model

probe = eval_stream.batch(torch.arange(16)).to(DEVICE)
with torch.no_grad():
    with autocast_context(DEVICE, BASELINE_COMPUTE):
        reference_logits, _ = target(probe)
reference_probs = torch.softmax(reference_logits.float(), dim=-1)
reference_top1 = reference_logits.argmax(-1)

print(f"{'формат':>7} {'эфф. бит/вес':>12} {'сжатие':>8} {'лосс':>8} {'Δлосс':>8} "
      f"{'KL, нат':>9} {'top-1 совп.':>12}")
print(f"{BASELINE_LABEL:>7} {BASELINE_STORAGE_BITS:>12.2f} {'×1.00':>8} {base_loss:>8.4f} {0.0:>8.4f} "
      f"{0.0:>9.5f} {1.0:>11.1%}")

quant_rows = []
for method in ("int8", "nvfp4", "mxfp4"):
    clone = copy.deepcopy(target)
    report = quantize_model(clone, method)
    loss = evaluate(clone, eval_stream, eval_cfg, batches=(2 if os.getenv("CAPSTONE_FAST_VALIDATION") == "1" else 16))
    with torch.no_grad():
        with autocast_context(DEVICE, BASELINE_COMPUTE):
            logits, _ = clone(probe)
    probs = torch.softmax(logits.float(), dim=-1)
    kl = torch.sum(reference_probs * (torch.log(reference_probs + 1e-12)
                                      - torch.log(probs + 1e-12)), dim=-1).mean().item()
    agreement = (logits.argmax(-1) == reference_top1).float().mean().item()
    quant_rows.append((method, report, loss, kl, agreement))
    print(f"{method:>7} {report['effective_bits_per_weight']:>12.2f} "
          f"{'×' + format(report['compression_vs_runtime_fp32'], '.2f'):>8} {loss:>8.4f} "
          f"{loss - base_loss:>+8.4f} {kl:>9.5f} {agreement:>11.1%}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
names = [BASELINE_LABEL] + [r[0] for r in quant_rows]
colors = ["#4C566A", "#A3BE8C", "#EBCB8B", "#BF616A"]

axes[0].bar(names, [base_loss] + [r[2] for r in quant_rows], color=colors)
axes[0].set_ylim(min([base_loss] + [r[2] for r in quant_rows]) * 0.98, max([base_loss] + [r[2] for r in quant_rows]) * 1.02)
axes[0].set_title("лосс (обрезанная ось!)"); axes[0].set_ylabel("нат")

axes[1].bar(names, [0.0] + [r[3] for r in quant_rows], color=colors)
axes[1].set_title("KL(baseline ‖ fake quant)"); axes[1].set_ylabel("нат")

axes[2].bar(names, [1.0] + [r[4] for r in quant_rows], color=colors)
axes[2].set_ylim(0, 1.05); axes[2].set_title("top-1 agreement с baseline")

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

Сравните все три графика и обратите внимание на начало оси loss. Близкие столбцы могут сочетаться с заметным изменением распределений. Упорядоченность форматов в одном запуске не является универсальным рейтингом: она зависит от весов, группировки, масштаба и задачи.

Здесь применяется fake quant: веса округляются и возвращаются в рабочий формат. Так проверяется ошибка представления без реализации упакованного вычислительного ядра. Полученные времена не доказывают ускорение четырёхбитного инференса; такое ускорение нужно измерять с соответствующим хранением и ядрами.

## 7. Спекулятивный декод

Черновик предлагает `k` токенов подряд, target проверяет их **одним** проходом.
Правило приёма `min(1, p/q)` с добором из нормированного остатка `max(0, p−q)`
сохраняет распределение target ровно (M11) — ускорение бесплатно по качеству.

Ключевая метрика — не доля приёма сама по себе, а **сколько токенов приходится
на один проход большой модели**. Без спекулятива она равна единице по
определению.

In [ ]:
from minillm.generate import speculative_generate

prompt = tokenizer.encode(corpus[:2000])[:32]
print(f"{'k':>3} {'приём':>8} {'токенов/проход target':>24} {'проходов draft':>16}")
lookahead_rows = []
for k in (1, 2, 4, 8):
    _, stats = speculative_generate(target, draft, prompt, 48, lookahead=k,
                                    sampling=SamplingConfig(temperature=1.0, seed=SEED))
    lookahead_rows.append((k, stats.acceptance_rate, stats.tokens_per_target_forward,
                           stats.draft_decode_forwards))
    print(f"{k:>3} {stats.acceptance_rate:>7.1%} "
          f"{stats.tokens_per_target_forward:>24.2f} {stats.draft_forwards:>16}")

In [ ]:
# Черновик, равный target: правило min(1, p/q) обязано принимать всё.
_, twin = speculative_generate(target, copy.deepcopy(target), prompt, 32, lookahead=4,
                               sampling=SamplingConfig(temperature=1.0, seed=SEED))
print(f"черновик == target: приём {twin.acceptance_rate:.1%}, "
      f"{twin.tokens_per_target_forward:.2f} токена на проход")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ks = [r[0] for r in lookahead_rows]
axes[0].plot(ks, [r[1] for r in lookahead_rows], "o-", color="#5E81AC")
axes[0].set_xlabel("глубина черновика k"); axes[0].set_ylabel("доля приёма")
axes[0].set_title("измеренная доля принятия")
axes[1].plot(ks, [r[2] for r in lookahead_rows], "o-", color="#A3BE8C")
axes[1].axhline(1.0, ls="--", lw=1, color="#4C566A", label="без спекулятива")
axes[1].set_xlabel("глубина черновика k")
axes[1].set_ylabel("токенов на проход target")
axes[1].set_title("выдача на проверочный проход"); axes[1].legend()
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

import json
inference_record = {"seed": SEED, "device": DEVICE,
    "compute_dtype": BASELINE_COMPUTE, "parameter_dtype": BASELINE_PARAMETER,
    "evaluation_sha": eval_stream.checksum(), "base_loss": base_loss,
    "quantization": [dict(method=r[0], report=r[1], loss=r[2], kl=r[3], top1=r[4]) for r in quant_rows],
    "speculative": [dict(k=r[0], acceptance=r[1], tokens_per_verification=r[2], draft_decode_passes=r[3]) for r in lookahead_rows],
    "identical_draft_acceptance": twin.acceptance_rate,
    "prefix_greedy_identical": identical}
Path("runs/inference").mkdir(parents=True, exist_ok=True)
Path("runs/inference/measurements.json").write_text(
    json.dumps(inference_record, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")

Доля принятия и токены на проверочный проход описывают разные стороны опыта. При постоянной вероятности принятия увеличение длины черновика повышает ожидаемое число выданных токенов с насыщением, но реальная вероятность зависит от позиции и истории. Ни обязательного падения acceptance, ни монотонности измеренной кривой из алгоритма не следует.

Принятые токены тоже требуют работы: черновик их предложил, target проверил. Сопоставьте сэкономленные проходы со стоимостью обеих моделей и не смешивайте prefill с проверкой. М11 выводит условие выгодности при заданной модели стоимости. Здесь батч равен единице, а серверная пропускная способность и зависимость от нагрузки не измеряются.

## 8. Итог и границы

Размер KV-кэша проверяется по геометрии и фактическому типу данных. Префиксный опыт проверяет повторное использование уже вычисленных состояний, а квантизация сравнивает несколько видов изменения предсказаний. Спекулятивный опыт считает принятие и выдачу токенов отдельно от затрат черновика.

Сохранённый отчёт описывает конкретное устройство, веса и входы. Небольшой замер не устанавливает аппаратное узкое место; fake quant не проверяет ускорение упакованного ядра; псевдоязык не заменяет целевые задачи. Для сравнения времени повторите прогоны после прогрева и укажите разброс. Для качества используйте отложенные примеры и разбор ошибок.

Далее ноутбук 4 связывает архитектурные изменения с контролируемыми опытами и строит итоговую таблицу из измерений своего запуска.